In [1]:
import pandas as pd
import numpy as np
from functions.running import prepare_data
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.decomposition import PCA
from functions.training import train
from functions.networks import SimpleNN, FullNN

In [2]:
df_train = pd.read_table('https://archive.ics.uci.edu/ml/machine-learning-databases/spect/SPECTF.train', header = None,sep=',')
df_test = pd.read_table('https://archive.ics.uci.edu/ml/machine-learning-databases/spect/SPECTF.test',
                     header=None, sep = ',')
df = pd.concat([df_train, df_test])
data = df.to_numpy()
X,y = data[:,1:], data[:,0]
G = len(np.unique(y))
for g in range(G):
  print(sum(y==g))
X = X.astype('float')

55
212


In [3]:
dataname = 'heart'
input_dim = X.shape[1]
epochs = 200
batch_size = 64
n_layer = 5
init_type = 'he'
dataname_ = dataname + str(n_layer) + init_type
missing = False

X_train, X_test, y_train, y_test = prepare_data(X,y, missing = missing)
input_dim = X_train.shape[1]  # Number of features

output_dim = len(np.unique(y_train))  # Number of classes (for Iris dataset)

variance_retained = .95
pca = PCA(n_components=variance_retained)
pca.fit(X_train)
n_components = pca.n_components_

hidden_dim = n_components  # Hidden layer size

other_layers = SimpleNN(input_dim=n_components, hidden_dim=hidden_dim, output_dim=output_dim, n_layer = n_layer, init_type = init_type)

train_loader = torch.utils.data.DataLoader(list(zip(X_train, y_train)), batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(list(zip(X_test, y_test)), batch_size=batch_size, shuffle=False)

criterion = nn.CrossEntropyLoss()
learning_rate = 0.01
n_frozen_epochs = 30

In [4]:
import torch.nn.functional as F

def predict_proba(model, X_pca):
    """
    Predict class probabilities using a trained neural network on PCA features.
    
    Args:
        model (nn.Module): Trained PyTorch model.
        X_pca (torch.Tensor): Input data in PCA space, shape (n_samples, n_components).
    
    Returns:
        torch.Tensor: Probabilities for each class, shape (n_samples, n_classes).
    """
    model.eval()
    with torch.no_grad():
        logits = model(X_pca)
        return F.softmax(logits, dim=1)

def predict(model, X_pca):
    """
    Predict class labels using a trained neural network on PCA features.
    
    Args:
        model (nn.Module): Trained PyTorch model.
        X_pca (torch.Tensor): Input data in PCA space.
    
    Returns:
        torch.Tensor: Predicted class labels.
    """
    probs = predict_proba(model, X_pca)
    return torch.argmax(probs, dim=1)


In [5]:
feature_names = [f'Feature{i}' for i in range(1, X_train.shape[1] + 1)]
columns = [f'PC{i}' for i in range(1, n_components+1)]

# Calculate the PCA loadings (components_)
loadings = pd.DataFrame(pca.components_.T[:,:len(columns)], columns=columns,
                        index=feature_names)  # feature_names needs to be defined as the list of feature names

loadings.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC14,PC15,PC16,PC17,PC18,PC19,PC20,PC21,PC22,PC23
Feature1,-0.091579,-0.147770,-0.107187,0.256573,0.151579,0.157323,0.137289,0.248182,0.227637,-0.154044,...,0.138069,-0.292347,0.194761,-0.326412,0.265317,-0.095593,-0.072158,-0.095789,0.162472,-0.140119
Feature2,-0.119804,-0.143954,-0.145595,0.202025,-0.011670,0.095188,0.079014,-0.214876,0.208980,-0.147673,...,0.250916,-0.284062,-0.117156,0.287822,0.060260,0.228470,-0.043819,0.089652,-0.141668,0.025019
Feature3,-0.109932,0.018026,-0.224853,0.108973,0.100714,0.009900,-0.262719,0.292090,0.374339,0.065854,...,-0.202628,0.246240,0.240627,0.107243,-0.288619,0.103968,-0.025719,0.075398,0.060490,-0.072208
Feature4,-0.112283,0.056748,-0.289328,0.163696,-0.083024,0.143049,-0.092168,0.077053,0.279165,0.088331,...,-0.238263,0.308986,0.108163,0.225029,0.231279,-0.105562,0.101100,-0.026396,-0.286517,0.116634
Feature5,-0.131152,0.069836,0.126680,0.191414,0.204398,0.220286,0.112427,-0.086861,0.078800,0.034151,...,-0.044775,-0.183269,0.217406,-0.156262,-0.225725,-0.167366,-0.033479,0.042456,-0.084514,0.066263


# SHAPLEY VALUES

## PCAFFW

### PCAFFW - TRAINING

In [6]:
X_train_t = torch.tensor(X_train)
X_test_t = torch.tensor(X_test)
y_train_t = torch.tensor(y_train, dtype=torch.long)

# ====== Interpret a simple neural network on Principal Components ======
simple_nn = other_layers
optimizer = optim.Adam(simple_nn.parameters(), lr=learning_rate)

# Apply PCA to training data
pca = PCA(n_components=n_components)
X_train_pca = pca.fit_transform(X_train.numpy())
X_test_pca = pca.transform(X_test.numpy())
X_train_pca, X_test_pca = torch.FloatTensor(X_train_pca), torch.FloatTensor(X_test_pca)

model_pcaffw = train(simple_nn,
                        torch.utils.data.DataLoader(list(zip(X_train_pca, y_train)), batch_size=batch_size, shuffle=True),
                        torch.utils.data.DataLoader(list(zip(X_test_pca, y_test)), batch_size=batch_size, shuffle=False),
                        criterion, optimizer, epochs=epochs)[0]

/var/folders/42/h0csfrkn2fvfq63027xhz7d40000gn/T/ipykernel_18161/3147366709.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train_t = torch.tensor(X_train)
/var/folders/42/h0csfrkn2fvfq63027xhz7d40000gn/T/ipykernel_18161/3147366709.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test_t = torch.tensor(X_test)
/var/folders/42/h0csfrkn2fvfq63027xhz7d40000gn/T/ipykernel_18161/3147366709.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train_t = torch.tensor(y_train, dtype=torch.long)


Epoch 1/200, Training Loss: 1.2738, Testing Accuracy: 0.8148, Training Time: 0.0228
Epoch 2/200, Training Loss: 0.5048, Testing Accuracy: 0.8148, Training Time: 0.0312
Epoch 3/200, Training Loss: 0.4580, Testing Accuracy: 0.8148, Training Time: 0.0392
Epoch 4/200, Training Loss: 0.4221, Testing Accuracy: 0.8148, Training Time: 0.0472
Epoch 5/200, Training Loss: 0.3916, Testing Accuracy: 0.8148, Training Time: 0.0550
Epoch 6/200, Training Loss: 0.3754, Testing Accuracy: 0.8148, Training Time: 0.0629
Epoch 7/200, Training Loss: 0.3551, Testing Accuracy: 0.8148, Training Time: 0.0709
Epoch 8/200, Training Loss: 0.3316, Testing Accuracy: 0.8148, Training Time: 0.0767
Epoch 9/200, Training Loss: 0.3094, Testing Accuracy: 0.8272, Training Time: 0.0855
Epoch 10/200, Training Loss: 0.2847, Testing Accuracy: 0.8519, Training Time: 0.0933
Epoch 11/200, Training Loss: 0.2576, Testing Accuracy: 0.8642, Training Time: 0.1011
Epoch 12/200, Training Loss: 0.2284, Testing Accuracy: 0.8765, Training Ti

### PCAFFW - SHAP

In [ ]:
import shap
print("\n=== SHAP PCAFFW===")
#pca_stage_model = nn.Sequential(model_pcsinit.flatten, model_pcsinit.fc1)  # only the PCA projection layer

background_np = X_train_pca[:50].numpy()
num_samples = 20
test_np = X_test_pca.numpy()

feature_names = [f'PC{i}' for i in range(1, n_components+1)]

def torch_predict_ffw(x_numpy):
    x_torch = torch.tensor(x_numpy, dtype=torch.float32)
    with torch.no_grad():
        return model_pcaffw(x_torch).numpy()
    
explainer_pca = shap.KernelExplainer(torch_predict_ffw, background_np)
shap_values_pca = explainer_pca.shap_values(test_np)



=== SHAP PCAFFW===


  0%|          | 0/20 [00:00<?, ?it/s]

In [8]:
import matplotlib.pyplot as plt
global_shap_values = shap_values_pca.mean(axis=0) # shape: (n_component x num_labels)

def get_10_important_PCs(shape_values, label):
    abs_vals = np.abs(shape_values[:, label])
    top10_idx = np.argsort(abs_vals)[-10:][::-1]  # largest 10
    PC_feature_names = [f"PC{i+1}" for i in top10_idx]
    global_importance = shape_values[top10_idx, label]  # shape (10,)
    # Put into DataFrame with feature names as columns
    global_importance_df = pd.DataFrame(
        [global_importance],  # one row of values
        columns=PC_feature_names
    )
    return global_importance_df, PC_feature_names

def get_important_PCs(shape_values, label):
    PC_feature_names = [f"PC{i+1}" for i in range(1,24)]
    # Put into DataFrame with feature names as columns
    df = pd.DataFrame(
        [shape_values],  # one row of values
        columns=PC_feature_names
    )
    return df, PC_feature_names


# Class=0
global_importance_0, PC_feature_names_0 = get_10_important_PCs(global_shap_values, 0)
loadings_top10_0 = loadings[PC_feature_names_0]
adjusted_importance_0 = loadings_top10_0.multiply(global_importance_0.values, axis=1).T

# Class=1
global_importance_1, PC_feature_names_1 = get_10_important_PCs(global_shap_values, 1)
loadings_top10_1 = loadings[PC_feature_names_1]
adjusted_importance_1 = loadings_top10_1.multiply(global_importance_1.values, axis=1).T


In [9]:
def get_important_PCs(shape_values, label):
    vals = shape_values[:, label]
    PC_feature_names = [f"PC{i}" for i in range(1,24)]
    # Put into DataFrame with feature names as columns
    df = pd.DataFrame(
        [vals],  # one row of values
        columns=PC_feature_names
    )
    return df, PC_feature_names

feature_SHAPE_1, PC_feature_names_1 = get_important_PCs(global_shap_values,1)
pca_SHAPE_1 = loadings.multiply(feature_SHAPE_1.values, axis=1).T
feature_SHAPE_0, PC_feature_names_0 = get_important_PCs(global_shap_values,0)
pca_SHAPE_0 = loadings.multiply(feature_SHAPE_0.values, axis=1).T

### PCAFFW - 10 most important Principal Components - global importance

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import shap
from matplotlib.colors import ListedColormap
    
cmap = ListedColormap(['#06923E', '#FFC107'], name='GgYy')

# Mean absolute SHAP value across samples for Class 0
ffw_importance_class0 = np.mean(np.abs(shap_values_pca[:, :, 0]), axis=0)

# Get indices of top 10 features
top10_idx_class0 = np.array(sorted(set(np.argsort(ffw_importance_class0)[-10:])))

# Feature names in the exact order we want
pca_feature_names_class0 = [f'PC_{i+1}' for i in top10_idx_class0]

# Create figure and axis
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

# Plot with fixed order
shap.summary_plot(
    shap_values_pca[:, top10_idx_class0, 0],
    test_np[:, top10_idx_class0],
    feature_names=pca_feature_names_class0,
    show=False,
    sort=False,   # <-- this is key
)

# Recolor the scatter points manually
ax = plt.gca()
for coll in ax.collections:
    coll.set_cmap(cmap)

# Title
ax.set_title('Class 0')

# Colorbar
fig = plt.gcf()
cbar = fig.axes[-1]
cbar.set_ylabel("PC value")
plt.savefig("plots/FFW_PC_global_class0.pdf", format="pdf", bbox_inches="tight")

plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import shap

# Mean absolute SHAP value across samples for Class 0
ffw_importance_class1 = np.mean(np.abs(shap_values_pca[:, :, 1]), axis=0)

# Get indices of top 10 features
#top10_idx_class1 = np.array(sorted(set(np.argsort(ffw_importance_class1)[-10:])))

# Feature names in the exact order we want
#pca_feature_names_class1 = [f'PC_{i+1}' for i in top10_idx_class1]

# Create figure and axis
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

# Plot with fixed order
shap.summary_plot(
    shap_values_pca[:, top10_idx_class0, 1],
    test_np[:, top10_idx_class0],
    feature_names=pca_feature_names_class0,
    show=False,
    sort=False,   # <-- this is key
)

# Title
ax.set_title('Class 1')

# Colorbar label
cbar = fig.axes[-1]
cbar.set_ylabel("PC value")
plt.savefig("plots/FFW_PC_global_class1.pdf", format="pdf", bbox_inches="tight")
plt.show()


### PCAFFW - the importance of 10 most important Features for explaining a certain sample

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt

def natural_sort_key(s):
    """Sorts strings like Feature7, Feature14 by their numeric part."""
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(r'(\d+)', s)]

# Values
values = pca_SHAPE_1.iloc[1]

# Top 10 by absolute value
top_idx = values.abs().nlargest(10).index

# Sort naturally
sorted_top_idx = sorted(top_idx, key=natural_sort_key)
sorted_top_idx = sorted_top_idx[::-1]

# Get values and colors
top_values = values[sorted_top_idx]
top_colors = ['#0D92F4' if v > 0 else '#FF0B55' for v in top_values]

# Plot
plt.figure(figsize=(12, 6))
bars = plt.barh(range(len(top_values)), top_values, color=top_colors)
plt.yticks(range(len(top_values)), sorted_top_idx)
plt.xlabel("SHAP Value")

# Add values with matching bar color
for i, (v, c) in enumerate(zip(top_values, top_colors)):
    offset = 0.02 * (1 if v > 0 else -1)  # adjust space from bar end
    plt.text(v + offset, i, f"{v:.3f}",
             va='center',
             ha='left' if v > 0 else 'right',
             color=c, fontsize=11)

# Add more margin on x-axis so labels don’t overlap
plt.xlim(min(top_values) - 0.2, max(top_values) + 0.2)
plt.title('Class 1')

plt.tight_layout()
plt.savefig("plots/FFW_local_PC2_class1.pdf", format="pdf", bbox_inches="tight")

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt

def natural_sort_key(s):
    """Sorts strings like Feature7, Feature14 by their numeric part."""
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(r'(\d+)', s)]

# Values
values = pca_SHAPE_0.iloc[1]

# Top 10 by absolute value
top_idx = values.abs().nlargest(10).index

# Sort naturally
sorted_top_idx = sorted(top_idx, key=natural_sort_key)
sorted_top_idx = sorted_top_idx[::-1]

# Get values and colors
top_values = values[sorted_top_idx]
top_colors = ['#0D92F4' if v > 0 else '#FF0B55' for v in top_values]

# Plot
plt.figure(figsize=(12, 6))
bars = plt.barh(range(len(top_values)), top_values, color=top_colors)
plt.yticks(range(len(top_values)), sorted_top_idx)
plt.xlabel("SHAP Value")

# Add values with matching bar color
for i, (v, c) in enumerate(zip(top_values, top_colors)):
    offset = 0.02 * (1 if v > 0 else -1)  # adjust space from bar end
    plt.text(v + offset, i, f"{v:.3f}",
             va='center',
             ha='left' if v > 0 else 'right',
             color=c, fontsize=11)

# Add more margin on x-axis so labels don’t overlap
plt.xlim(min(top_values) - 0.2, max(top_values) + 0.2)
plt.title('Class 0')

plt.tight_layout()
plt.savefig("plots/FFW_local_PC2_class0.pdf", format="pdf", bbox_inches="tight")

## PCSINIT

### PCSINIT - TRAINING

In [299]:
print("Training with PCA-initialized NN...")
pca_init_nn = FullNN(input_dim, n_components, other_layers, activation='none', init_type=init_type)
pca_init_nn.init_pca_weights(X_train)  # Initialize weights with PCA components

# train on everything except the first layer
optimizer = optim.Adam([{'params': param} for name, param in pca_init_nn.named_parameters() if not name.startswith('fc1')],
                        lr=learning_rate)

model_pcsinit = train(pca_init_nn, train_loader, test_loader, criterion, optimizer, epochs=n_frozen_epochs)[0]
model_pcsinit_output = train(pca_init_nn, train_loader, test_loader, criterion, optimizer, epochs=n_frozen_epochs)[1]


Training with PCA-initialized NN...
Number of PCA components: 23
Epoch 1/30, Training Loss: 6.4174, Testing Accuracy: 0.7778, Training Time: 0.0250
Epoch 2/30, Training Loss: 2.1902, Testing Accuracy: 0.8148, Training Time: 0.0349
Epoch 3/30, Training Loss: 1.2514, Testing Accuracy: 0.8148, Training Time: 0.0445
Epoch 4/30, Training Loss: 0.9038, Testing Accuracy: 0.8025, Training Time: 0.0608
Epoch 5/30, Training Loss: 0.5172, Testing Accuracy: 0.7778, Training Time: 0.0717
Epoch 6/30, Training Loss: 0.3290, Testing Accuracy: 0.7407, Training Time: 0.0810
Epoch 7/30, Training Loss: 0.3244, Testing Accuracy: 0.6667, Training Time: 0.0910
Epoch 8/30, Training Loss: 0.2904, Testing Accuracy: 0.6914, Training Time: 0.1000
Epoch 9/30, Training Loss: 0.2402, Testing Accuracy: 0.7407, Training Time: 0.1072
Epoch 10/30, Training Loss: 0.2227, Testing Accuracy: 0.7531, Training Time: 0.1166
Epoch 11/30, Training Loss: 0.2048, Testing Accuracy: 0.7407, Training Time: 0.1237
Epoch 12/30, Trainin

### PCSINIT - SHAP

In [ ]:
# ====== Interpret Full Model After Training ======

print("\n=== SHAP PCSINIT ===")
background_np = X_train.numpy()
test_np = X_test.numpy()

def pcsinit_torch_predict(x_numpy):
    x_torch = torch.tensor(x_numpy, dtype=torch.float32)
    with torch.no_grad():
        return model_pcsinit(x_torch).numpy()

pcsinit_explainer = shap.KernelExplainer(pcsinit_torch_predict, background_np)
pcsinit_shap_values = pcsinit_explainer.shap_values(test_np)


=== SHAP PCSINIT ===


  0%|          | 0/20 [00:00<?, ?it/s]

### PCSINIT - Top 10 important Components - Global Importance

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import shap

# Mean absolute SHAP value across samples for Class 0
pcsinit_importance_class0 = np.mean(np.abs(pcsinit_shap_values[:, :, 0]), axis=0)

# Get indices of top 10 features
top10_idx_class0 = np.array(sorted(set(np.argsort(pcsinit_importance_class0)[-10:])))

# Feature names in the exact order we want
pcsinit_feature_names_class0 = [f'Feature_{i+1}' for i in top10_idx_class0]

# Create figure and axis
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

# Plot with fixed order
shap.summary_plot(
    pcsinit_shap_values[:, top10_idx_class0, 0],
    test_np[:, top10_idx_class0],
    feature_names=pcsinit_feature_names_class0,
    show=False,
    sort=False,   # <-- this is key
)

# Title
ax.set_title('Class 0')

# Colorbar label
cbar = fig.axes[-1]
cbar.set_ylabel("Feature value")
plt.savefig("plots/PCSINIT_feature_global_class0.pdf", format="pdf", bbox_inches="tight")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import shap

# Mean absolute SHAP value across samples for Class 1
pcsinit_importance_class1 = np.mean(np.abs(pcsinit_shap_values[:, :, 1]), axis=0)

# Get indices of top 10 features
#top10_idx_class0 = np.array(sorted(set(np.argsort(pcsinit_importance_class0)[-10:])))

# Feature names in the exact order we want
#pcsinit_feature_names_class0 = [f'Feature_{i+1}' for i in top10_idx_class0]

# Create figure and axis
fig, ax = plt.subplots(1, 1, figsize=(14, 6))

# Plot with fixed order
shap.summary_plot(
    pcsinit_shap_values[:, top10_idx_class0, 1],
    test_np[:, top10_idx_class0],
    feature_names=pcsinit_feature_names_class0,
    show=False,
    sort=False,   # <-- this is key
)

# Title
ax.set_title('Class 1')

# Colorbar label
cbar = fig.axes[-1]
cbar.set_ylabel("Feature value")
plt.savefig("plots/PCSINIT_feature_global_class1.pdf", format="pdf", bbox_inches="tight")


### PCSINIT - Feature Importance of 1 certain sample

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sample_idx = 1   # pick your sample
class_idx = 0    # pick your class

# Get SHAP values for that sample and class
shap_sample = pcsinit_shap_values[sample_idx, :, class_idx]

# Find top 10 features by absolute SHAP value
top_idx = np.argsort(np.abs(shap_sample))[-10:][::-1]

# Sort in ascending feature index order
top_idx = sorted(top_idx)

top_features = [f"Feature{i+1}" for i in top_idx]
top_values = shap_sample[top_idx]

# Color bars
colors = ['#0D92F4' if v > 0 else '#FF0B55' for v in top_values]

# Plot
plt.figure(figsize=(10, 6))
bars = plt.barh(range(10), top_values, color=colors)
plt.yticks(range(10), top_features)
plt.xlabel("SHAP value (impact on model output)")
plt.gca().invert_yaxis()

# Add text labels
for i, (v, c) in enumerate(zip(top_values, colors)):
    offset = 0.02 * (1 if v > 0 else -1)
    plt.text(v + offset, i, f"{v:.3f}",
             va='center',
             ha='left' if v > 0 else 'right',
             color=c)

# Expand margins
x_min = min(top_values) - abs(min(top_values)) * 0.1
x_max = max(top_values) + abs(max(top_values)) * 0.1
plt.xlim(x_min, x_max)

plt.title('Class 0')
plt.tight_layout(pad=2)  # add padding for extra breathing room

plt.savefig(
    "plots/PCSINIT_local_class0.pdf",
    format="pdf", bbox_inches="tight"
)


In [ ]:
class_idx = 1    # pick your class

# Get SHAP values for that sample and class
shap_sample = pcsinit_shap_values[sample_idx, :, class_idx]

top_values = shap_sample[top_idx]

# Color bars
colors = ['#0D92F4' if v > 0 else '#FF0B55' for v in top_values]

# Plot
plt.figure(figsize=(10, 6))
bars = plt.barh(range(10), top_values, color=colors)
plt.yticks(range(10), top_features)
plt.xlabel("SHAP value (impact on model output)")
plt.gca().invert_yaxis()

# Add text labels
for i, (v, c) in enumerate(zip(top_values, colors)):
    offset = 0.02 * (1 if v > 0 else -1)
    plt.text(v + offset, i, f"{v:.3f}",
             va='center',
             ha='left' if v > 0 else 'right',
             color=c)

# Expand margins
x_min = min(top_values) - abs(min(top_values)) * 0.1
x_max = max(top_values) + abs(max(top_values)) * 0.1
plt.xlim(x_min, x_max)

plt.title('Class 1')
plt.tight_layout(pad=2)  # add padding for extra breathing room

plt.savefig(
    "plots/PCSINIT_local_class1.pdf",
    format="pdf", bbox_inches="tight"
)
